# ESMC Mutation Scoring - Colab Version

Score individual mutations using ESMC zero-shot mutation scoring with leave-one-out masking.

**What it does:**
- Masks each position one at a time and runs inference
- Computes Shannon entropy (conservation at each position)
- Computes log-likelihood ratios (effect of each possible mutation)
- Identifies mutation-tolerant positions for protein engineering

**Key outputs:**
- Entropy plot (constrained vs. flexible positions)
- Deleterious fraction (% of 20 AAs that are harmful at each position)
- LLR heatmap (20 AAs × sequence positions)
- CSV with per-position mutation scores


In [ ]:
!pip install -q torch transformers accelerate einops biotite biopython scikit-learn pandas numpy tqdm ipywidgets matplotlib

In [ ]:
!pip install -q git+https://github.com/Biohub/esm.git@main

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import json
from tqdm import tqdm
from IPython.display import display, Markdown, clear_output

# ESM imports
from esm.models import ESMCForCausalLM
from esm.tokenization import TokenizerBase

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
# Model & device
MODEL_NAME = "esmc_600m"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4  # Batch masked positions

# Canonical amino acids (for mutation scoring)
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

# Output
OUTPUT_DIR = "/content/mutation_scoring_output"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"Canonical AAs: {len(AMINO_ACIDS)} ({AMINO_ACIDS})")

## Load ESMC Model

In [ ]:
print(f"Loading {MODEL_NAME}...")
model = ESMCForCausalLM.from_pretrained(f"esm2/esmc_{MODEL_NAME}").to(DEVICE)
model.eval()
tokenizer = TokenizerBase()
print(f"✓ Model loaded. Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## Utility Functions

In [ ]:
def load_fasta(filepath: str) -> Dict[str, str]:
    """Load FASTA file."""
    sequences = {}
    with open(filepath) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            sequences[current_id] = "".join(current_seq)
    return sequences

def get_leave_one_out_logits(model, tokenizer, sequence: str, device, batch_size: int = 4):
    """Get logits with each position masked.
    
    Returns list of logit arrays, one per position.
    """
    logits_list = []
    
    # Create masked sequences (one mask per position)
    masked_sequences = [
        sequence[:i] + "_" + sequence[i + 1:]
        for i in range(len(sequence))
    ]
    
    # Process in batches
    with torch.no_grad():
        for batch_start in range(0, len(masked_sequences), batch_size):
            batch_end = min(batch_start + batch_size, len(masked_sequences))
            batch_seqs = masked_sequences[batch_start:batch_end]
            
            # Tokenize batch
            tokens_batch = [tokenizer.tokenize(seq) for seq in batch_seqs]
            
            # Pad to same length
            max_len = max(len(t) for t in tokens_batch)
            tokens_padded = [
                t + [tokenizer.pad_id] * (max_len - len(t))
                for t in tokens_batch
            ]
            tokens_tensor = torch.tensor(tokens_padded).to(device)
            
            # Forward pass
            output = model(tokens_tensor)
            logits = output.logits.cpu().numpy()
            
            # Extract logits at masked position (position i+1 accounting for BOS)
            for j, seq_idx in enumerate(range(batch_start, batch_end)):
                # Position in sequence + 1 for BOS token
                pos = seq_idx + 1
                logits_list.append(logits[j, pos, :])
    
    return np.array(logits_list)  # [seq_len, vocab_size]

def compute_entropy(logits: np.ndarray) -> np.ndarray:
    """Compute Shannon entropy from logits.
    
    Args:
        logits: [seq_len, vocab_size] array
    
    Returns:
        entropy: [seq_len] array (bits)
    """
    logits_torch = torch.from_numpy(logits).float()
    probs = F.softmax(logits_torch, dim=-1).numpy()
    entropy = -np.sum(probs * np.log2(probs + 1e-10), axis=-1)
    return entropy

def compute_log_likelihood_ratios(logits: np.ndarray, sequence: str) -> np.ndarray:
    """Compute LLR for each position and AA.
    
    Args:
        logits: [seq_len, vocab_size] array
        sequence: Wild-type sequence
    
    Returns:
        llr: [seq_len, 20] array (canonical AAs)
    """
    logits_torch = torch.from_numpy(logits).float()
    log_probs = F.log_softmax(logits_torch, dim=-1)
    
    llr_list = []
    for i, wt_aa in enumerate(sequence):
        if wt_aa not in AA_TO_IDX:
            # Non-standard AA, skip
            llr_list.append(np.zeros(len(AMINO_ACIDS)))
            continue
        
        wt_idx = AA_TO_IDX[wt_aa]
        wt_logprob = log_probs[i, wt_idx].item()
        
        # Compute LLR for each canonical AA
        llr_row = []
        for aa_idx in range(len(AMINO_ACIDS)):
            aa_logprob = log_probs[i, aa_idx].item()
            llr = aa_logprob - wt_logprob
            llr_row.append(llr)
        llr_list.append(llr_row)
    
    return np.array(llr_list)  # [seq_len, 20]

def compute_fraction_deleterious(llr: np.ndarray) -> np.ndarray:
    """Fraction of AAs with negative LLR at each position.
    
    Returns:
        fraction: [seq_len] array, 0.0-1.0
    """
    return np.mean(llr < 0, axis=1)

## Load Sequence

In [ ]:
#@markdown Upload FASTA or paste sequence

from google.colab import files

input_mode = "paste"  # "upload" or "paste"

if input_mode == "upload":
    print("Upload your FASTA file:")
    uploaded = files.upload()
    fasta_file = list(uploaded.keys())[0]
    sequences = load_fasta(fasta_file)
else:
    # Example: GFP
    sequences = {
        "GFP": "MVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVAWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"
    }

print(f"Loaded {len(sequences)} sequences")
for seq_id, seq in sequences.items():
    print(f"  {seq_id}: {len(seq)} aa")

## Compute Mutation Scores

In [ ]:
mutation_results = {}

for seq_id, sequence in sequences.items():
    print(f"\n{'='*60}")
    print(f"Computing mutation scores for {seq_id} ({len(sequence)} aa)")
    print(f"{'='*60}")
    
    # Get leave-one-out logits
    print("Step 1: Computing leave-one-out logits...")
    logits = get_leave_one_out_logits(model, tokenizer, sequence, DEVICE, BATCH_SIZE)
    print(f"  ✓ Logits shape: {logits.shape}")
    
    # Compute entropy
    print("Step 2: Computing entropy...")
    entropy = compute_entropy(logits)
    print(f"  ✓ Mean entropy: {np.mean(entropy):.2f} bits")
    print(f"  ✓ Range: {np.min(entropy):.2f} - {np.max(entropy):.2f} bits")
    
    # Compute LLRs
    print("Step 3: Computing log-likelihood ratios...")
    llr = compute_log_likelihood_ratios(logits, sequence)
    print(f"  ✓ LLR shape: {llr.shape}")
    print(f"  ✓ Mean LLR: {np.mean(llr):.3f}")
    
    # Compute deleterious fraction
    print("Step 4: Computing deleterious fraction...")
    frac_del = compute_fraction_deleterious(llr)
    print(f"  ✓ Mean fraction deleterious: {np.mean(frac_del):.2f}")
    
    mutation_results[seq_id] = {
        "sequence": sequence,
        "logits": logits,
        "entropy": entropy,
        "llr": llr,
        "fraction_deleterious": frac_del
    }

print(f"\n✓ Completed mutation scoring for {len(mutation_results)} sequences")

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

for seq_id, data in mutation_results.items():
    entropy = data["entropy"]
    sequence = data["sequence"]
    
    fig, ax = plt.subplots(figsize=(14, 4))
    
    # Plot entropy
    ax.plot(entropy, linewidth=2, color='steelblue', label='Entropy')
    ax.fill_between(range(len(entropy)), entropy, alpha=0.3, color='steelblue')
    
    # Reference lines for uniform distributions
    ax.axhline(y=np.log2(2), color='gray', linestyle='--', alpha=0.5, label='Uniform 2 AAs')
    ax.axhline(y=np.log2(4), color='gray', linestyle=':', alpha=0.5, label='Uniform 4 AAs')
    ax.axhline(y=np.log2(20), color='red', linestyle='-', alpha=0.3, linewidth=2, label='Uniform 20 AAs (max)')
    
    ax.set_xlabel('Sequence Position')
    ax.set_ylabel('Shannon Entropy (bits)')
    ax.set_title(f'{seq_id} - Entropy Analysis (Conservation Landscape)')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    # Annotate 10 lowest entropy positions
    lowest_entropy_idx = np.argsort(entropy)[:10]
    for idx in lowest_entropy_idx:
        ax.annotate(
            sequence[idx],
            xy=(idx, entropy[idx]),
            xytext=(0, -15),
            textcoords='offset points',
            fontsize=8,
            ha='center'
        )
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{seq_id}_entropy.png", dpi=150, bbox_inches='tight')
    plt.show()

print("✓ Entropy plots saved")

In [ ]:
for seq_id, data in mutation_results.items():
    frac_del = data["fraction_deleterious"]
    
    fig, ax = plt.subplots(figsize=(14, 4))
    
    # Scatter plot
    ax.scatter(range(len(frac_del)), frac_del, alpha=0.6, s=30, color='coral')
    
    # Highlight mutation-tolerant positions (fraction < 0.8)
    tolerant_idx = np.where(frac_del < 0.8)[0]
    if len(tolerant_idx) > 0:
        ax.scatter(tolerant_idx, frac_del[tolerant_idx], alpha=0.8, s=50, color='green', label='Mutation-tolerant (<80%)')
    
    # Threshold line
    ax.axhline(y=0.8, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Threshold (80%)')
    
    ax.set_xlabel('Sequence Position')
    ax.set_ylabel('Fraction of Deleterious Mutations')
    ax.set_title(f'{seq_id} - Mutation Tolerance (Fraction Deleterious)')
    ax.set_ylim([-0.05, 1.05])
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{seq_id}_deleterious_fraction.png", dpi=150, bbox_inches='tight')
    plt.show()

print("✓ Deleterious fraction plots saved")

In [ ]:
for seq_id, data in mutation_results.items():
    llr = data["llr"]
    sequence = data["sequence"]
    
    fig, ax = plt.subplots(figsize=(16, 5))
    
    # Heatmap
    im = ax.imshow(llr.T, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2, interpolation='nearest')
    
    # Set ticks
    ax.set_yticks(range(len(AMINO_ACIDS)))
    ax.set_yticklabels(AMINO_ACIDS)
    ax.set_xlabel('Sequence Position')
    ax.set_ylabel('Amino Acid')
    ax.set_title(f'{seq_id} - Log-Likelihood Ratios (Mutation Scores) Heatmap')
    
    # Mark wildtype positions with dots
    for i, wt_aa in enumerate(sequence):
        if wt_aa in AA_TO_IDX:
            aa_idx = AA_TO_IDX[wt_aa]
            ax.plot(i, aa_idx, 'k.', markersize=5, alpha=0.7)
    
    plt.colorbar(im, ax=ax, label='Log-Likelihood Ratio (nats)')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{seq_id}_llr_heatmap.png", dpi=150, bbox_inches='tight')
    plt.show()

print("✓ LLR heatmaps saved")

## Top Mutation Recommendations

In [ ]:
for seq_id, data in mutation_results.items():
    llr = data["llr"]
    sequence = data["sequence"]
    entropy = data["entropy"]
    
    print(f"\n{'='*80}")
    print(f"Top Mutation Recommendations for {seq_id}")
    print(f"{'='*80}")
    
    # Find best mutations (high LLR) at high-entropy (flexible) positions
    mutations = []
    for pos in range(len(sequence)):
        wt_aa = sequence[pos]
        if wt_aa not in AA_TO_IDX:
            continue
        
        wt_idx = AA_TO_IDX[wt_aa]
        
        # Find best mutations at this position
        for mut_idx, mut_aa in enumerate(AMINO_ACIDS):
            if mut_idx != wt_idx:  # Only non-wildtype
                llr_score = llr[pos, mut_idx]
                mutations.append({
                    'mutation': f"{wt_aa}{pos+1}{mut_aa}",
                    'position': pos + 1,
                    'wt_aa': wt_aa,
                    'mut_aa': mut_aa,
                    'llr': llr_score,
                    'entropy': entropy[pos]
                })
    
    # Sort by LLR (best mutations have highest positive LLR)
    df_mutations = pd.DataFrame(mutations)
    df_mutations = df_mutations.sort_values('llr', ascending=False)
    
    # Top beneficial mutations
    print("\nTop 20 Beneficial Mutations (highest LLR):")
    print(df_mutations.head(20)[['mutation', 'llr', 'entropy']].to_string(index=False))
    
    # Top deleterious mutations
    print("\nTop 20 Deleterious Mutations (lowest LLR):")
    print(df_mutations.tail(20)[['mutation', 'llr', 'entropy']].to_string(index=False))
    
    # Save all mutations
    df_mutations.to_csv(f"{OUTPUT_DIR}/{seq_id}_mutations.csv", index=False)

## Save Results

In [ ]:
# Save per-position metrics
for seq_id, data in mutation_results.items():
    df_metrics = pd.DataFrame({
        "position": range(1, len(data["sequence"]) + 1),
        "wildtype_aa": list(data["sequence"]),
        "entropy": data["entropy"],
        "fraction_deleterious": data["fraction_deleterious"]
    })
    df_metrics.to_csv(f"{OUTPUT_DIR}/{seq_id}_per_position_metrics.csv", index=False)

print(f"✓ Results saved to {OUTPUT_DIR}/")
print(f"  - Per-position metrics: {{seq_id}}_per_position_metrics.csv")
print(f"  - All mutations: {{seq_id}}_mutations.csv")
print(f"  - Entropy plot: {{seq_id}}_entropy.png")
print(f"  - Deleterious fraction: {{seq_id}}_deleterious_fraction.png")
print(f"  - LLR heatmap: {{seq_id}}_llr_heatmap.png")

## Interpretation Guide

In [ ]:
display(Markdown("""
## Interpreting Results

### **Entropy (Bits)**
- **Low entropy (< 1.0)**: Highly constrained position, risky to mutate
  - Example: Active site residues, critical folds
  - Mutation likely deleterious
- **Medium entropy (1.0-3.0)**: Moderately constrained
  - Some mutations may be tolerated
- **High entropy (> 3.5)**: Flexible position, mutation-tolerant
  - Good targets for engineering
  - Many mutations likely neutral/beneficial

### **Log-Likelihood Ratio (LLR)**
- **Positive LLR**: Mutation is more likely than wildtype
  - LLR > 1.0: Strong candidate for beneficial mutation
  - Model prefers this amino acid at this position
- **LLR = 0**: Wildtype (by definition)
- **Negative LLR**: Mutation less likely than wildtype
  - LLR < -1.0: Likely deleterious
  - Model prefers the current amino acid

### **Deleterious Fraction**
- Fraction of 20 canonical amino acids with negative LLR
- **0.2 (20%)**: Only 4 AAs are harmful, 16 are tolerated → good mutation target
- **0.8 (80%)**: 16 AAs are harmful, only 4 are tolerated → constrained position
- **1.0 (100%)**: All mutations are harmful → essential position

### **Design Strategy**
1. Find high-entropy positions (flexible, tolerant)
2. Look for mutations with high positive LLR
3. Prioritize changes from bulky to small or vice versa
4. Consider multiple mutations at tolerant positions
"""))